In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

In [ ]:
print("SQuAD Dataset yüklənir...")
raw_datasets = load_dataset("rajpurkar/squad")


train_data = raw_datasets["train"].select(range(0, 4000))
val_data = raw_datasets["validation"].select(range(0, 400))

SQuAD Dataset yüklənir...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]

    # Sual və konteksti birlikdə tokenizasiya edirik
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second", # Əgər limit keçilərsə, sualı yox, yalnız mətni kəsir
        return_offsets_mapping=True, # Simvolların mətndəki yerini tapmaq üçün
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        answer = answers[i]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Mətn hissəsinin harada başlayıb bitdiyini tapırıq
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # Cavab mətndən kənarda qalıbsa (0, 0) qeyd edirik
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Başlanğıc token indeksini təyin edirik
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            # Bitiş token indeksini təyin edirik
            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Funksiyanı dataya tətbiq edirik
print("Data hazırlanır...")
tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)

Data hazırlanır...


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:
data_collator = DefaultDataCollator()

training_args = TrainingArguments(
    output_dir="./bert-qa-squad",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Təlim başlayır...")
trainer.train()

Təlim başlayır...


Epoch,Training Loss,Validation Loss
1,2.145380,1.911518
2,1.310104,1.637047
3,1.064411,1.637360


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=1.897211456298828, metrics={'train_runtime': 153.1858, 'train_samples_per_second': 78.336, 'train_steps_per_second': 9.792, 'total_flos': 1175877900288000.0, 'train_loss': 1.897211456298828, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./saved_qa_model")
tokenizer.save_pretrained("./saved_qa_model")
print("QA Modeli uğurla yadda saxlanıldı!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

QA Modeli uğurla yadda saxlanıldı!


In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

# 1. Modeli və Tokenizer-i yükləyirik
model_dir = "./saved_qa_model"
model = AutoModelForQuestionAnswering.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

context_text = "Azerbaijan State University of Economics, known as UNEC, is a public university located in Baku."
question_text = "Where is UNEC located?"


inputs = tokenizer(question_text, context_text, return_tensors="pt", max_length=384, truncation=True)


with torch.no_grad():
    outputs = model(**inputs)


start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits)


predict_answer_tokens = inputs.input_ids[0][start_idx : end_idx + 1]
answer = tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)

print("Sual:", question_text)
print("Modelin Cavabı:", answer)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Sual: Where is UNEC located?
Modelin Cavabı: azerbaijan state university of economics, known as unec, is a public university located in baku
